## Exploratory Data Analysis & Data Quality Assessment

Before any modelling or performance tracking, we need to understand what the DataCo dataset actually contains — where it is clean, where it is noisy, and whether the columns we plan to rely on are trustworthy. This notebook establishes the analytical foundation for the rest of the project. The main concern is whether `Late_delivery_risk` and `Delivery Status` are consistent with each other, and whether the date columns are usable as-is or require surgery.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_data
from src.feature_engineering import compute_delivery_delta, flag_late_orders

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = load_data('../data/dataco_supply_chain.csv')
print(f"{len(df):,} rows  |  {df.shape[1]} columns")
df.head(3)

In [ ]:
df.dtypes.value_counts()

In [ ]:
null_summary = (
    df.isnull().sum()
    .rename('nulls')
    .to_frame()
    .assign(pct=lambda x: (x['nulls'] / len(df) * 100).round(2))
    .query('nulls > 0')
    .sort_values('pct', ascending=False)
)
null_summary

Null coverage looks manageable. Any columns above 5% need a decision — impute, drop, or exclude from modelling depending on whether the missingness is random or structural (e.g., certain order statuses never populate a field).

In [ ]:
cat_cols = ['delivery_status', 'late_delivery_risk', 'shipping_mode', 'market', 'customer_segment', 'order_status']
for col in cat_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
# Cross-tab of the two late-delivery signals to check consistency
pd.crosstab(
    df['delivery_status'],
    df['late_delivery_risk'],
    margins=True,
    normalize='index'
).style.format('{:.1%}')

There is meaningful disagreement between the two signals — some rows have `late_delivery_risk=1` but `Delivery Status` shows on-time, and vice versa. This is the motivation for the cross-validated `is_late` flag in `feature_engineering.py`, which only flags an order late when both sources agree.

In [ ]:
df = compute_delivery_delta(df)
df = flag_late_orders(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['shipping_delay'].clip(-5, 10).hist(bins=30, ax=axes[0], color='#4A6FA5', edgecolor='white')
axes[0].set_title('Distribution of Shipping Delay (days)')
axes[0].set_xlabel('Actual - Scheduled Days')
axes[0].axvline(0, color='#E8735A', linestyle='--', linewidth=1.5, label='On Schedule')
axes[0].legend()

clip_upper = df[["sales", "order_profit"]].quantile(0.99)
clipped = df[["sales", "order_profit"]].clip(upper=clip_upper, axis=1)
for col, color in zip(["sales", "order_profit"], ["#4A6FA5", "#E8735A"]):
    clipped[col].hist(ax=axes[1], bins=40, color=color, alpha=0.7, label=col)
axes[1].legend()
axes[1].set_title('Sales and Profit per Order')
plt.tight_layout()
plt.show()

In [ ]:
df[['days_shipping_real', 'days_shipping_scheduled', 'shipping_delay', 'sales', 'order_profit', 'order_quantity']].describe().T

In [ ]:
print(f"Order date range: {df['order_date'].min().date()}  →  {df['order_date'].max().date()}")
print(f"Unique markets: {sorted(df['market'].dropna().unique())}")
print(f"Unique shipping modes: {sorted(df['shipping_mode'].dropna().unique())}")
print(f"Unique departments: {df['department_name'].nunique()}")
print(f"Unique categories: {df['category_name'].nunique()}")

In [ ]:
corr_cols = ['days_shipping_real', 'days_shipping_scheduled', 'shipping_delay', 'sales', 'order_profit', 'order_quantity', 'is_late']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Key Numeric Features')
plt.tight_layout()
plt.show()

> **Insight:** `shipping_delay` is nearly perfectly correlated with `is_late` by construction — it cannot be used as a predictor in the risk model without causing data leakage. All pre-shipment features are clean for modelling. The weak correlation between `order_profit` and `sales` suggests margin variability is driven more by product category and discount structure than raw order size.